# PrioritAI — Conformal Calibration Notebook
## Generating `q_hat` for the ResNet50 Damage Classifier

---

### What this notebook does

This is an **offline calibration step**. It does not retrain or modify the model.

Using a labeled calibration dataset, it computes `q_hat` — the conformal threshold that gives a **coverage guarantee**: with `alpha = 0.05`, the prediction set will contain the true damage class at least **95% of the time** on data drawn from the same distribution.

### Architecture context

```
Frontend → Backend (FastAPI) → ML Service (FastAPI + Keras)
                                         ↑
                               [Conformal wrapper — future]
                               reads q_hat.json at startup
```

### Class ordering

The model was trained with folders sorted **alphabetically**:

| Index | Class  | damage_score |
|-------|--------|--------------|
| 0     | heavy  | 7            |
| 1     | light  | 3            |
| 2     | medium | 5            |

### Output

`q_hat.json` saved to Google Drive — loaded by the runtime service in a future step.

---
## Step 1 — Mount Google Drive

All paths below are relative to your Drive root. Edit the `DRIVE_ROOT` variable in Step 2 if your files are in a subfolder.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## Step 2 — Configuration

Edit these paths to match your Google Drive layout before running the rest of the notebook.

```
MyDrive/
├── rocket_damage_resnet50_v2.keras   ← MODEL_PATH
├── labeled_data/                      ← DATA_ROOT
│   ├── heavy/
│   ├── light/
│   └── medium/
└── conformal_output/
    └── q_hat.json                     ← OUTPUT_PATH (created automatically)
```

In [17]:
# ── Google Drive paths ────────────────────────────────────────────────────────
DRIVE_ROOT  = "/content/drive/MyDrive"
MODEL_PATH  = f"{DRIVE_ROOT}/Project/rocket_damage_resnet50_v2.keras"
DATA_ROOT   = f"{DRIVE_ROOT}/Project/data_splits/ResNet50_2/val"
OUTPUT_PATH = f"{DRIVE_ROOT}/Project/conformal_output/q_hat.json"

# ── Must match production ml-service/app/inference.py ────────────────────────
IMG_SIZE    = (224, 224)      # resize target
CLASS_NAMES = ["light", "medium", "heavy"]

# ── Conformal calibration ─────────────────────────────────────────────────────
ALPHA       = 0.15            # target miscoverage rate → 95 % coverage guarantee
BATCH_SIZE  = 32              # images per model.predict() call

print("Configuration:")
print(f"  MODEL_PATH  : {MODEL_PATH}")
print(f"  DATA_ROOT   : {DATA_ROOT}")
print(f"  OUTPUT_PATH : {OUTPUT_PATH}")
print(f"  IMG_SIZE    : {IMG_SIZE}")
print(f"  CLASS_NAMES : {CLASS_NAMES}")
print(f"  ALPHA       : {ALPHA}")

Configuration:
  MODEL_PATH  : /content/drive/MyDrive/Project/rocket_damage_resnet50_v2.keras
  DATA_ROOT   : /content/drive/MyDrive/Project/data_splits/ResNet50_2/val
  OUTPUT_PATH : /content/drive/MyDrive/Project/conformal_output/q_hat.json
  IMG_SIZE    : (224, 224)
  CLASS_NAMES : ['light', 'medium', 'heavy']
  ALPHA       : 0.15


---
## Step 3 — Imports

All libraries are pre-installed in Google Colab. No `pip install` needed.

In [3]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pillow     : {Image.__version__}")

TensorFlow : 2.20.0
NumPy      : 2.0.2
Pillow     : 11.3.0


---
## Step 4 — Load the Keras Model

The model is loaded identically to production `preload_model()` in `ml-service/app/inference.py`:

```python
model = tf.keras.models.load_model(MODEL_PATH)
```

A warmup predict runs immediately after loading to force graph compilation, matching runtime behaviour.

In [4]:
print(f"Loading model from:\n  {MODEL_PATH}\n")

if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(
        f"Model not found at {MODEL_PATH}\n"
        "Check that MODEL_PATH in Step 2 points to the correct Drive location."
    )

model = tf.keras.models.load_model(MODEL_PATH)

# Warmup pass — identical to production preload_model()
dummy = np.zeros((1, *IMG_SIZE, 3), dtype=np.float32)
_ = model.predict(dummy, verbose=0)
print("Warmup complete.\n")

model.summary()

Loading model from:
  /content/drive/MyDrive/Project/rocket_damage_resnet50_v2.keras

Warmup complete.



Model: "Rocket_Damage_Classifier_ResNet50_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,180,043 (96.05 MB)

 Trainable params: 529,411 (2.02 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

 Optimizer params: 1,058,824 (4.04 MB)

---
## Step 5 — Preprocessing Pipeline

This function replicates the **exact** preprocessing from `ml-service/app/inference.py`:

```python
img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
img = img.resize(IMG_SIZE)                          # 224 × 224
arr = np.array(img, dtype=np.float32) / 255.0      # normalise [0, 1]
arr = np.expand_dims(arr, axis=0)                   # add batch dim
```

The only difference here is that we read from a file path instead of bytes, and we do not add the batch dimension yet (that happens during batched inference in Step 6).

In [5]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess_image(image_path: str) -> np.ndarray:
    """
    Preprocessing must exactly match training and production inference.
    """

    img = Image.open(image_path).convert("RGB")
    img = img.resize(IMG_SIZE)

    img_array = np.array(img).astype(np.float32)

    # ResNet50 preprocessing
    arr = preprocess_input(img_array)

    return arr

print("preprocess_image() defined.")
print(f"Output shape per image: {IMG_SIZE + (3,)}")

preprocess_image() defined.
Output shape per image: (224, 224, 3)


---
## Step 6 — Load Calibration Dataset

Images are loaded from the three class folders. Folder names are sorted **alphabetically** to assign class indices — this must match how the model was trained:

| Folder | Index |
|--------|-------|
| heavy  | 0     |
| light  | 1     |
| medium | 2     |

Supported image extensions: `.jpg`, `.jpeg`, `.png`, `.bmp`, `.webp`

In [6]:
import random

CLASS_TO_IDX     = {name: idx for idx, name in enumerate(CLASS_NAMES)}
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

MAX_PER_CLASS = 200

image_paths  = []   # List[Path]
image_labels = []   # List[int]

print("Scanning calibration dataset...")
print(f"  Root    : {DATA_ROOT}")
print(f"  Classes : {CLASS_NAMES}  (indices 0-{len(CLASS_NAMES)-1})\n")

for class_name in CLASS_NAMES:
    class_dir = Path(DATA_ROOT) / class_name

    if not class_dir.is_dir():
        print(f"  [WARN] Folder not found, skipping: {class_dir}")
        continue

    files = sorted(
        f for f in class_dir.iterdir()
        if f.suffix.lower() in VALID_EXTENSIONS
    )

    # Random sampling if too many images
    if len(files) > MAX_PER_CLASS:
        files = random.sample(files, MAX_PER_CLASS)

    image_paths.extend(files)
    image_labels.extend([CLASS_TO_IDX[class_name]] * len(files))

    print(f"  [{CLASS_TO_IDX[class_name]}] {class_name:8s}  →  {len(files):4d} images")

print(f"\nTotal: {len(image_paths)} images")

if len(image_paths) == 0:
    raise RuntimeError(
        "No images found. Check that DATA_ROOT in Step 2 points to the "
        "correct Drive folder and that image files exist inside the class subfolders."
    )

Scanning calibration dataset...
  Root    : /content/drive/MyDrive/Project/data_splits/ResNet50_2/val
  Classes : ['light', 'medium', 'heavy']  (indices 0-2)

  [0] light     →   200 images
  [1] medium    →   200 images
  [2] heavy     →   200 images

Total: 600 images


---
## Step 7 — Run Model Inference on All Calibration Images

Images are processed in batches of `BATCH_SIZE = 32` to avoid running out of memory.

The output is two arrays:
- `calibration_probs` — shape `(n, 3)`, raw softmax probabilities for each image
- `calibration_labels` — shape `(n,)`, integer ground-truth class index for each image

These are the inputs to the conformal calibration algorithm in Step 8.

In [7]:
n           = len(image_paths)
all_probs   = []
load_errors = 0

print(f"Running inference on {n} images (batch_size={BATCH_SIZE})...\n")
t0 = time.time()

for batch_start in range(0, n, BATCH_SIZE):
    batch_paths  = image_paths[batch_start : batch_start + BATCH_SIZE]
    batch_arrays = []

    for path in batch_paths:
        try:
            arr = preprocess_image(str(path))
            batch_arrays.append(arr)
        except Exception as exc:
            print(f"  [ERROR] {path.name}: {exc}")
            load_errors += 1

            # Zero-fill so array indices stay aligned with image_labels
            batch_arrays.append(
                np.zeros((*IMG_SIZE, 3), dtype=np.float32)
            )

    batch_tensor = np.stack(batch_arrays, axis=0)          # (B, 224, 224, 3)

    batch_probs = model.predict(batch_tensor, verbose=0)   # (B, num_classes)

    # Safety check — verify model output matches expected number of classes
    if batch_probs.shape[1] != len(CLASS_NAMES):
        raise ValueError(
            f"Model output shape mismatch: got {batch_probs.shape[1]} classes "
            f"but CLASS_NAMES has {len(CLASS_NAMES)}"
        )

    all_probs.append(batch_probs)

    done    = min(batch_start + BATCH_SIZE, n)
    elapsed = time.time() - t0
    pct     = done / n * 100

    print(f"  [{done:4d}/{n}]  {pct:5.1f}%   {elapsed:5.1f}s elapsed")

calibration_probs  = np.concatenate(all_probs, axis=0)        # (n, 3)
calibration_labels = np.array(image_labels, dtype=np.int32)   # (n,)

print(f"\nInference complete.")
print(f"  calibration_probs  shape : {calibration_probs.shape}")
print(f"  calibration_labels shape : {calibration_labels.shape}")

if load_errors > 0:
    print(f"\n[WARN] {load_errors} image(s) failed to load and were replaced with zeros.")
    print("       Review the errors above — they may affect calibration quality.")

# Quick sanity: probabilities should sum to ~1 per row
row_sums = calibration_probs.sum(axis=1)

print(
    f"\nSoftmax row-sum check: "
    f"min={row_sums.min():.4f}  "
    f"max={row_sums.max():.4f}  "
    f"(expect ~1.0)"
)

# Class distribution sanity check
print("\nClass distribution:")
for class_name, idx in CLASS_TO_IDX.items():
    count = np.sum(calibration_labels == idx)
    print(f"  [{idx}] {class_name:8s} → {count} images")

Running inference on 600 images (batch_size=32)...

  [  32/600]    5.3%    25.4s elapsed
  [  64/600]   10.7%    42.7s elapsed
  [  96/600]   16.0%    60.3s elapsed
  [ 128/600]   21.3%    81.1s elapsed
  [ 160/600]   26.7%   105.9s elapsed
  [ 192/600]   32.0%   126.3s elapsed
  [ 224/600]   37.3%   150.3s elapsed
  [ 256/600]   42.7%   169.0s elapsed
  [ 288/600]   48.0%   190.0s elapsed
  [ 320/600]   53.3%   209.6s elapsed
  [ 352/600]   58.7%   230.8s elapsed
  [ 384/600]   64.0%   250.0s elapsed
  [ 416/600]   69.3%   272.5s elapsed
  [ 448/600]   74.7%   294.1s elapsed
  [ 480/600]   80.0%   315.5s elapsed
  [ 512/600]   85.3%   334.0s elapsed
  [ 544/600]   90.7%   359.8s elapsed
  [ 576/600]   96.0%   378.9s elapsed
  [ 600/600]  100.0%   396.6s elapsed

Inference complete.
  calibration_probs  shape : (600, 3)
  calibration_labels shape : (600,)

Softmax row-sum check: min=1.0000  max=1.0000  (expect ~1.0)

Class distribution:
  [0] light    → 200 images
  [1] medium   → 200

---
## Step 8 — Conformal Calibration Algorithm

### Theory

Conformal Prediction constructs a **prediction set** instead of a single label. The set is guaranteed to contain the true class with at least `1 - alpha` probability, without any distributional assumptions beyond exchangeability.

**Nonconformity score** for image `i`:
$$s_i = 1 - \hat{p}_{y_i}$$
where $\hat{p}_{y_i}$ is the model's softmax probability for the **true** class $y_i$.

A high score means the model was uncertain or wrong about the true class.

**Calibration quantile:**
$$\hat{q} = \text{Quantile}\left(s_1, \ldots, s_n,\ \frac{\lceil (n+1)(1-\alpha) \rceil}{n}\right)$$

The `+1` finite-sample correction ensures exact marginal coverage.

**At runtime**, the prediction set for a new image is:
$$\mathcal{C}(x) = \{ k : \hat{p}_k \geq 1 - \hat{q} \}$$

### Implementation

The cell below contains the **exact algorithm** that will also be used in the runtime ML service. Do not modify it.

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# APS (Adaptive Prediction Sets) conformal implementation
# More robust for overconfident classifiers
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np

def compute_conformal_threshold(
    calibration_probs,
    calibration_labels,
    alpha=0.05,
):
    n = len(calibration_probs)

    if n == 0:
        return 1.0

    scores = []

    for probs, true_label in zip(calibration_probs, calibration_labels):

        # Sort probabilities descending
        sorted_indices = np.argsort(probs)[::-1]
        sorted_probs   = probs[sorted_indices]

        # Find where the true label appears
        true_rank = np.where(sorted_indices == true_label)[0][0]

        # APS score = cumulative probability up to true label
        cumulative_prob = np.sum(sorted_probs[:true_rank + 1])

        scores.append(cumulative_prob)

    scores = np.array(scores)

    q_level = np.ceil((n + 1) * (1 - alpha)) / n
    q_level = min(q_level, 1.0)

    q_hat = np.quantile(scores, q_level, method="higher")

    return q_hat


def predict_conformal(
    model_prediction,
    q_hat,
    class_names,
):

    sorted_indices = np.argsort(model_prediction)[::-1]

    prediction_set = []
    cumulative_prob = 0.0

    for idx in sorted_indices:

        prediction_set.append(class_names[idx])

        cumulative_prob += model_prediction[idx]

        if cumulative_prob >= q_hat:
            break

    return prediction_set


print("APS conformal functions defined:")
print("  compute_conformal_threshold(calibration_probs, calibration_labels, alpha)")
print("  predict_conformal(model_prediction, q_hat, class_names)")

APS conformal functions defined:
  compute_conformal_threshold(calibration_probs, calibration_labels, alpha)
  predict_conformal(model_prediction, q_hat, class_names)


---
## Step 9 — Compute `q_hat`

Runs `compute_conformal_threshold` over the full calibration set.

In [20]:
q_hat = compute_conformal_threshold(
    calibration_probs,
    calibration_labels,
    alpha=ALPHA,
)

threshold = 1.0 - q_hat

print("Conformal calibration complete")
print(f"  n (calibration images) : {len(calibration_probs)}")
print(f"  alpha                  : {ALPHA}  (target miscoverage rate)")
print(f"  coverage guarantee     : {(1 - ALPHA) * 100:.0f}%")
print(f"  q_hat                  : {q_hat:.6f}")
print(f"  runtime threshold      : {threshold:.6f}")
print()
print(f"Interpretation: a class is included in the prediction set")
print(f"when its softmax probability >= {threshold:.4f}")

Conformal calibration complete
  n (calibration images) : 600
  alpha                  : 0.15  (target miscoverage rate)
  coverage guarantee     : 85%
  q_hat                  : 1.000000
  runtime threshold      : 0.000000

Interpretation: a class is included in the prediction set
when its softmax probability >= 0.0000


In [21]:
print("Sample APS prediction sets:\n")

for i in range(20):

    probs = calibration_probs[i]

    pred_set = predict_conformal(
        probs,
        q_hat,
        CLASS_NAMES,
    )

    pred_class = CLASS_NAMES[np.argmax(probs)]

    print(
        f"true={CLASS_NAMES[calibration_labels[i]]:7s} | "
        f"pred={pred_class:7s} | "
        f"set={pred_set}"
    )

Sample APS prediction sets:

true=light   | pred=medium  | set=['medium', 'light']
true=light   | pred=light   | set=['light']
true=light   | pred=medium  | set=['medium', 'light']
true=light   | pred=light   | set=['light', 'medium', 'heavy']
true=light   | pred=light   | set=['light', 'medium']
true=light   | pred=medium  | set=['medium', 'light']
true=light   | pred=light   | set=['light', 'medium']
true=light   | pred=heavy   | set=['heavy']
true=light   | pred=medium  | set=['medium', 'heavy', 'light']
true=light   | pred=light   | set=['light', 'medium']
true=light   | pred=medium  | set=['medium', 'light']
true=light   | pred=medium  | set=['medium', 'light', 'heavy']
true=light   | pred=light   | set=['light', 'medium', 'heavy']
true=light   | pred=light   | set=['light']
true=light   | pred=light   | set=['light', 'medium']
true=light   | pred=light   | set=['light', 'medium', 'heavy']
true=light   | pred=light   | set=['light', 'medium', 'heavy']
true=light   | pred=heavy   |

In [22]:
correct = 0
set_sizes = []

for probs, true_label in zip(calibration_probs, calibration_labels):

    pred_set = predict_conformal(
        probs,
        q_hat,
        CLASS_NAMES,
    )

    true_class = CLASS_NAMES[true_label]

    # Coverage
    if true_class in pred_set:
        correct += 1

    # Set size
    set_sizes.append(len(pred_set))

coverage = correct / len(calibration_labels)
mean_set_size = np.mean(set_sizes)

print("APS Evaluation:\n")

print(f"Coverage        : {coverage:.4f}")
print(f"Mean set size   : {mean_set_size:.4f}")

print("\nSet size distribution:")

for size in sorted(set(set_sizes)):
    count = set_sizes.count(size)
    print(f"  size {size}: {count}")

APS Evaluation:

Coverage        : 0.9183
Mean set size   : 1.8233

Set size distribution:
  size 1: 254
  size 2: 198
  size 3: 148


---
## Step 10 — Save `q_hat.json`

The output file is self-documenting: it records the threshold value alongside the calibration metadata so the runtime service can validate it on load.

In [23]:
output_dir = Path(OUTPUT_PATH).parent
output_dir.mkdir(parents=True, exist_ok=True)

payload = {
    "q_hat":          float(q_hat),
    "alpha":          ALPHA,
    "n_calibration":  int(len(calibration_probs)),
    "class_names":    CLASS_NAMES,
    "model_path":     MODEL_PATH,
    "date_saved":     datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Saved → {OUTPUT_PATH}\n")
print(json.dumps(payload, indent=2))

Saved → /content/drive/MyDrive/Project/conformal_output/q_hat.json

{
  "q_hat": 1.0,
  "alpha": 0.15,
  "n_calibration": 600,
  "class_names": [
    "light",
    "medium",
    "heavy"
  ],
  "model_path": "/content/drive/MyDrive/Project/rocket_damage_resnet50_v2.keras",
  "date_saved": "2026-05-26T09:01:04Z"
}


---
## Step 11 — Sanity Checks

Three checks to confirm the calibration is working correctly:

1. **Sample predictions** — show prediction sets for 10 random calibration images and whether the true class is covered.
2. **Full calibration set coverage** — must be ≥ `1 - alpha` (95 %).
3. **Mean prediction set size** — indicates model certainty. Values close to 1 mean the model is confident; values close to 3 mean it is uncertain on most images.

In [24]:
print("=" * 65)
print("SANITY CHECK 1 — prediction sets on 10 random calibration images")
print("=" * 65)

rng     = np.random.default_rng(42)
indices = rng.choice(len(calibration_probs), size=min(10, len(calibration_probs)), replace=False)

covered_sample = 0
for i in indices:
    probs      = calibration_probs[i]
    true_label = CLASS_NAMES[calibration_labels[i]]
    pred_set   = predict_conformal(probs, q_hat, CLASS_NAMES)
    is_covered = true_label in pred_set
    covered_sample += int(is_covered)
    marker = "OK" if is_covered else "MISS"
    print(
        f"  [{marker:4s}]  true={true_label:8s}  "
        f"probs={[round(p, 3) for p in probs]}  "
        f"set={pred_set}"
    )

print(f"\nSample coverage: {covered_sample}/{len(indices)} = {covered_sample / len(indices):.0%}")

# ── Full calibration set coverage ────────────────────────────────────────────
print()
print("=" * 65)
print("SANITY CHECK 2 — coverage over full calibration set")
print("=" * 65)

covered_full = sum(
    CLASS_NAMES[calibration_labels[i]] in predict_conformal(
        calibration_probs[i], q_hat, CLASS_NAMES
    )
    for i in range(len(calibration_probs))
)
full_coverage = covered_full / len(calibration_probs)
status = "PASS" if full_coverage >= (1 - ALPHA) else "FAIL"
print(f"  Coverage : {full_coverage:.4f}  (target >= {1 - ALPHA:.2f})  [{status}]")

# ── Mean prediction set size ──────────────────────────────────────────────────
print()
print("=" * 65)
print("SANITY CHECK 3 — prediction set size distribution")
print("=" * 65)

set_sizes = [
    len(predict_conformal(calibration_probs[i], q_hat, CLASS_NAMES))
    for i in range(len(calibration_probs))
]
size_counts = {s: set_sizes.count(s) for s in sorted(set(set_sizes))}

for size, count in size_counts.items():
    bar = "#" * int(count / len(set_sizes) * 40)
    print(f"  size {size}: {count:5d} images ({count / len(set_sizes):.1%})  {bar}")

print(f"\n  Mean set size : {np.mean(set_sizes):.3f}")
print(f"  (1.0 = model always certain, {len(CLASS_NAMES)}.0 = model always uncertain)")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"  q_hat          : {q_hat:.6f}")
print(f"  alpha          : {ALPHA}")
print(f"  n_calibration  : {len(calibration_probs)}")
print(f"  coverage       : {full_coverage:.4f}  [{status}]")
print(f"  mean set size  : {np.mean(set_sizes):.3f}")
print(f"  output file    : {OUTPUT_PATH}")

SANITY CHECK 1 — prediction sets on 10 random calibration images
  [OK  ]  true=light     probs=[np.float32(1.0), np.float32(0.0), np.float32(0.0)]  set=['light']
  [OK  ]  true=heavy     probs=[np.float32(0.0), np.float32(0.0), np.float32(1.0)]  set=['heavy']
  [OK  ]  true=light     probs=[np.float32(0.864), np.float32(0.017), np.float32(0.118)]  set=['light', 'heavy', 'medium']
  [OK  ]  true=medium    probs=[np.float32(0.013), np.float32(0.987), np.float32(0.0)]  set=['medium', 'light']
  [OK  ]  true=medium    probs=[np.float32(0.999), np.float32(0.001), np.float32(0.0)]  set=['light', 'medium', 'heavy']
  [OK  ]  true=medium    probs=[np.float32(0.998), np.float32(0.002), np.float32(0.0)]  set=['light', 'medium']
  [OK  ]  true=heavy     probs=[np.float32(0.0), np.float32(0.996), np.float32(0.004)]  set=['medium', 'heavy']
  [OK  ]  true=light     probs=[np.float32(0.216), np.float32(0.784), np.float32(0.0)]  set=['medium', 'light', 'heavy']
  [OK  ]  true=light     probs=[np.flo

---
## Next Steps

### What to do with `q_hat.json`

1. **Download** the file from Google Drive to your local machine.
2. **Place it** inside the ML service model directory:
   ```
   ml-service/model/q_hat.json
   ```
3. **In a future step**, the runtime `inference.py` will load `q_hat.json` at startup and use `predict_conformal()` to return prediction sets alongside the existing classification output.

### Expected `q_hat.json` structure

```json
{
  "q_hat": 0.412300,
  "alpha": 0.05,
  "n_calibration": 450,
  "class_names": ["heavy", "light", "medium"],
  "model_path": "/content/drive/MyDrive/rocket_damage_resnet50_v2.keras",
  "date_saved": "2026-05-25T10:00:00Z"
}
```

### Important notes

- The `class_names` list order in `q_hat.json` must match the model's output index order.
- `q_hat` is specific to the `alpha` value used. If you change `alpha`, re-run the notebook.
- Calibration should be re-run whenever the model is updated.